# Chapter 26
## Phase Locking of Two Oscillators
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter26.ipynb)

## About this chapter

Two pulse-coupled oscillators can be studied through an event-to-event phase
map instead of their full voltage trajectories: record only each cell's
phase in its cycle at the instant the other cell fires. If $g$ is a
phase-resetting curve, a single pulse maps phase $\varphi$ to
$f(\varphi)=\varphi+g(\varphi)$. Accounting for the other oscillator's
complementary phase gives $F(\varphi)=f(1-\varphi)$, and composing $F$ with
itself gives the two-event phase-difference map

$$
G(\varphi)=F(F(\varphi)).
$$

A locking relationship between the two oscillators is a fixed point
$G(\varphi_*)=\varphi_*$, and it is locally stable when $|G'(\varphi_*)|<1$.
The examples below plot $G$ against the identity line so those intersections
and their slopes can be read off directly.

The first five examples build this map from abstract PRCs $g(\varphi)$ with
symmetric or asymmetric shape, showing that symmetry of a locking
relationship is a property of the model, not a general guarantee. The
remaining examples iterate a pair of pulse-coupled oscillators directly and
compute the same map $G$ from the RTM conductance-based neuron's measured
PRC, connecting the abstract picture to a concrete model.

See [`README.md`](chapter26.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
from numba import njit

## Abstract Pulse-Coupling Maps (shared by the five examples below)

`abstract_phase_grid` is the common phase grid $\varphi\in[0,1]$.
`pulse_map_f`/`pulse_map_bigF`/`pulse_map_bigG` build $f=\varphi+g(\varphi)$,
$F(\varphi)=f(1-\varphi)$, and the two-event map $G=F\circ F$ from any PRC
$g$; `plot_pulse_coupling_full` shows all four ($g,f,F,G$, used when $f$'s
monotonicity is worth checking), while `plot_pulse_coupling_g_and_bigG`
shows just $g$ and $G$.

In [ ]:
def abstract_phase_grid(n=101):
    return np.arange(n) / (n - 1)


def pulse_map_f(g, phi):
    return phi + g(phi)


def pulse_map_bigF(g, phi):
    return pulse_map_f(g, 1 - phi)


def pulse_map_bigG(g, phi):
    return pulse_map_bigF(g, pulse_map_bigF(g, phi))


def check_f_monotonic(g, phi):
    '''Print a warning if f=phi+g(phi) is not strictly increasing on phi.'''
    f_vals = pulse_map_f(g, phi)
    if np.min(f_vals[1:] - f_vals[:-1]) <= 0:
        print('f is not strictly increasing')


def plot_pulse_coupling_full(g, phi):
    '''2x2 grid: g, f=phi+g(phi), F(phi)=f(1-phi), G=F(F(phi)).'''
    check_f_monotonic(g, phi)
    f_vals = pulse_map_f(g, phi)
    bigF_vals = pulse_map_bigF(g, phi)
    bigG_vals = pulse_map_bigG(g, phi)

    fig, axes = plt.subplots(2, 2, figsize=(8, 8))

    axes[0, 0].plot(phi, g(phi), '-k', linewidth=2)
    axes[0, 0].axis([0, 1, 0, 1])
    axes[0, 0].set_box_aspect(1)
    axes[0, 0].set_xlabel(r'$\varphi$')
    axes[0, 0].set_ylabel('$g$')

    axes[0, 1].plot(phi, f_vals, '-k', linewidth=2)
    axes[0, 1].axis([0, 1, 0, 1])
    axes[0, 1].set_box_aspect(1)
    axes[0, 1].set_xlabel(r'$\varphi$')
    axes[0, 1].set_ylabel('$f$')

    axes[1, 0].plot(phi, bigF_vals, '-k', linewidth=2)
    axes[1, 0].axis([0, 1, 0, 1])
    axes[1, 0].set_box_aspect(1)
    axes[1, 0].set_xlabel(r'$\varphi$')
    axes[1, 0].set_ylabel('$F$')

    axes[1, 1].plot(phi, bigG_vals, '-k', linewidth=2)
    axes[1, 1].plot([0, 1], [0, 1], '--k', linewidth=1)
    axes[1, 1].axis([0, 1, 0, 1])
    axes[1, 1].set_box_aspect(1)
    axes[1, 1].set_xlabel(r'$\varphi$')
    axes[1, 1].set_ylabel('$G$')

    plt.tight_layout()
    plt.show()


def plot_pulse_coupling_g_and_bigG(g, phi, g_ylim=(0, 1)):
    '''1x2 grid: g and G=F(F(phi)).'''
    bigG_vals = pulse_map_bigG(g, phi)

    fig, axes = plt.subplots(1, 2, figsize=(8, 4.5))

    axes[0].plot(phi, g(phi), '-k', linewidth=2)
    axes[0].axis([0, 1, g_ylim[0], g_ylim[1]])
    axes[0].set_box_aspect(1)
    axes[0].set_xlabel(r'$\varphi$')
    axes[0].set_ylabel('$g$')

    axes[1].plot(phi, bigG_vals, '-k', linewidth=2)
    axes[1].plot([0, 1], [0, 1], '--k', linewidth=1)
    axes[1].axis([0, 1, 0, 1])
    axes[1].set_box_aspect(1)
    axes[1].set_xlabel(r'$\varphi$')
    axes[1].set_ylabel('$G$')

    plt.tight_layout()
    plt.show()

### Abstract Pulse Coupling 1: Asymmetric Positive PRC

$g(\varphi)=\varphi^2(1-\varphi)$ is a positive, asymmetric PRC. Plotting
$g$, $f$, $F$, and $G$ together shows how the map's fixed points arise from
composing $f$ with itself after the phase flip.

In [ ]:
def g_pulse_1(phi):
    return phi ** 2 * (1 - phi)


phi = abstract_phase_grid()
plot_pulse_coupling_full(g_pulse_1, phi)

### Abstract Pulse Coupling 2: Steeper Asymmetric Response

$g(\varphi)=\epsilon\,\varphi(1-\varphi)^3$ with $\epsilon=2$ changes the
abstract response's shape to show a different fixed-point arrangement of
$G$.

In [ ]:
def g_pulse_2(phi, epsilon=2.0):
    return epsilon * phi * (1 - phi) ** 3


phi = abstract_phase_grid()
plot_pulse_coupling_g_and_bigG(g_pulse_2, phi, g_ylim=(0, 1))

In [ ]:
interact(lambda epsilon=2.0: plot_pulse_coupling_g_and_bigG(
    lambda phi: g_pulse_2(phi, epsilon=epsilon), abstract_phase_grid(), g_ylim=(0, 1)),
    epsilon=(0.5, 4.0, 0.1));

### Abstract Pulse Coupling 3: Antisymmetric PRC

$g(\varphi)=-\sin(2\pi\varphi)\big(\varphi(1-\varphi)+0.025\big)/2$ is
antisymmetric about $\varphi=1/2$ and gives its own pulse-coupling map
$G$.

In [ ]:
def g_pulse_3(phi):
    return -np.sin(2 * np.pi * phi) * (phi * (1 - phi) + 0.025) / 2


phi = abstract_phase_grid()
plot_pulse_coupling_g_and_bigG(g_pulse_3, phi, g_ylim=(-0.5, 0.5))

### Abstract Pulse Coupling 4: Asymmetric PRC, Shifted Locking

$g(\varphi)=2(1-\varphi)-\big(1+\epsilon-\sqrt{(1+\epsilon)^2-4\epsilon(1-\varphi)}\big)/\epsilon$
with $\epsilon=0.75$ is asymmetric and shifts the locking geometry away from
the symmetric case.

In [ ]:
def g_pulse_4(phi, epsilon=0.75):
    return 2 * (1 - phi) - (1 + epsilon - np.sqrt((1 + epsilon) ** 2 - 4 * epsilon * (1 - phi))) / epsilon


phi = abstract_phase_grid()
plot_pulse_coupling_full(g_pulse_4, phi)

In [ ]:
interact(lambda epsilon=0.75: plot_pulse_coupling_full(
    lambda phi: g_pulse_4(phi, epsilon=epsilon), abstract_phase_grid()),
    epsilon=(0.1, 1.5, 0.05));

### Abstract Pulse Coupling 5: Negative Asymmetric Response

$g(\varphi)=-\varphi^2(1-\varphi)$ is a further asymmetric response example,
the negative counterpart of the first one.

In [ ]:
def g_pulse_5(phi):
    return -phi ** 2 * (1 - phi)


phi = abstract_phase_grid()
plot_pulse_coupling_g_and_bigG(g_pulse_5, phi, g_ylim=(-1, 0))

## Transformed Map $\tilde f$

`simulate_f_tilde` plots the transformed map $\tilde f$ used to locate
phase-map fixed points geometrically: rotating $(\varphi,f)$ coordinates by
45 degrees into $(s,\tilde f)$ turns a fixed point of $f$ into a zero
crossing of $\tilde f$, and the annotated figure shows how the perpendicular
offset $u$ reconstructs $(\varphi,f(\varphi))$ from $(s,\tilde f(s))$.

In [ ]:
def simulate_f_tilde(n=101):
    def f_tilde(s):
        return 0.5 * (s + 1 / np.sqrt(2)) * (1 / np.sqrt(2) - s)

    s = np.zeros(n)
    phi = np.zeros(n)
    f = np.zeros(n)
    for k in range(n):
        s[k] = -1 / np.sqrt(2) + k / (n - 1) * np.sqrt(2)
        u = f_tilde(s[k]) / np.sqrt(2)
        phi[k] = 0.5 + s[k] / np.sqrt(2) - u
        f[k] = 0.5 + s[k] / np.sqrt(2) + u
    return s, phi, f


def plot_f_tilde(s, phi, f):
    plt.figure(figsize=(6, 6))
    plt.plot(phi, f, '-k', linewidth=6)
    plt.axis([0, 1, 0, 1])
    plt.gca().set_box_aspect(1)
    plt.xlabel(r'$\varphi$')
    plt.ylabel('$f$')
    plt.plot([0, 1], [0, 1], '--k', linewidth=2)

    epsilon = 0.025
    plt.plot([0.5 - epsilon, 0.5 + epsilon], [0.5 + epsilon, 0.5 - epsilon], '-k', linewidth=1)
    plt.plot([0.7 - epsilon, 0.7 + epsilon], [0.7 + epsilon, 0.7 - epsilon], '-k', linewidth=1)
    plt.text(0.55, 0.43, '$0$', fontsize=20, rotation=45)
    plt.text(0.75, 0.63, '$s$', fontsize=20, rotation=45)
    delta = 0.15
    plt.plot([0.7, 0.7 - delta], [0.7, 0.7 + delta], '-r', linewidth=4)
    plt.plot([0.7, 0.7 - delta], [0.7, 0.7], '-b', linewidth=2)
    plt.plot([0.7 - delta, 0.7 - delta], [0.7, 0.7 + delta], '-b', linewidth=2)
    plt.text(0.49, 0.75, '$u$', color='b', fontsize=20)
    plt.text(0.59, 0.67, '$u$', color='b', fontsize=20)
    plt.text(0.67, 0.8, r'$\tilde{f}$', color='r', fontsize=20)
    plt.plot(0.7 - delta, 0.7 + delta, '.k', markersize=20)
    plt.text(0.7 - delta - 0.43, 0.7 + delta + 0.05, r'$(\varphi,f(\varphi))$', fontsize=20)

    plt.tight_layout()
    plt.show()


plot_f_tilde(*simulate_f_tilde())

## Two Pulse-Coupled Oscillators

Rather than reading fixed points off $G$, iterate the two oscillators A and
B directly: whichever fires resets to phase 0 while the other jumps to
$f(1-\varphi)$ using its own PRC $g$. `simulate_two_pulse_coupled_osc` runs
this event-driven iteration for `N` cycles and returns each cell's spike
times; `plot_two_pulse_coupled_osc` shows them as a raster (A in red, B in
blue) so the emerging locking pattern is visible directly.

In [ ]:
def g_two_pulse_1(phi):
    return phi ** 2 * (1 - phi)


def g_two_pulse_2(phi):
    return 2 * phi * (1 - phi) ** 3


def simulate_two_pulse_coupled_osc(g, phi_A0=0., phi_B0=0.5, N=12):
    def f(phi):
        return phi + g(phi)

    phi_A, phi_B = phi_A0, phi_B0
    num_spikes_A, num_spikes_B = 1, 0
    t_spikes_A = [0.]
    t_spikes_B = []

    t = 0.
    for k in range(N):
        t = t + (1 - phi_B)
        num_spikes_B += 1
        t_spikes_B.append(t)
        phi_A = f(1 - phi_B)
        phi_B = 0.
        t = t + (1 - phi_A)
        num_spikes_A += 1
        t_spikes_A.append(t)
        phi_B = f(1 - phi_A)
        phi_A = 0.

    return np.array(t_spikes_A), np.array(t_spikes_B), num_spikes_A, num_spikes_B


def plot_two_pulse_coupled_osc(t_spikes_A, t_spikes_B, num_spikes_A, num_spikes_B, N=12):
    plt.figure(figsize=(8, 4))
    plt.plot(t_spikes_A, np.ones(num_spikes_A), '.r', markersize=15)
    plt.plot(t_spikes_B, 2 * np.ones(num_spikes_B), '.b', markersize=15)
    plt.axis([0, N - 2, 0, 3])
    plt.xticks(range(1, N))
    plt.yticks([])
    plt.xlabel('$t$ [units of $T$]')
    plt.title('spike times of A (red) and B (blue)')
    plt.tight_layout()
    plt.show()

### Two Pulse-Coupled Oscillators: Example 1

Starting from $\varphi_A=0,\varphi_B=0.5$ with the asymmetric PRC
$g(\varphi)=\varphi^2(1-\varphi)$.

In [ ]:
t_spikes_A, t_spikes_B, num_spikes_A, num_spikes_B = simulate_two_pulse_coupled_osc(
    g_two_pulse_1, phi_A0=0., phi_B0=0.5)
plot_two_pulse_coupled_osc(t_spikes_A, t_spikes_B, num_spikes_A, num_spikes_B)

In [ ]:
interact(lambda phi_B0=0.5: plot_two_pulse_coupled_osc(
    *simulate_two_pulse_coupled_osc(g_two_pulse_1, phi_A0=0., phi_B0=phi_B0)),
    phi_B0=(0.0, 1.0, 0.05));

### Two Pulse-Coupled Oscillators: Example 2

Repeats the two-cell iteration for a second pulse-response choice,
$g(\varphi)=2\varphi(1-\varphi)^3$, starting from
$\varphi_A=0,\varphi_B=0.05$.

In [ ]:
t_spikes_A, t_spikes_B, num_spikes_A, num_spikes_B = simulate_two_pulse_coupled_osc(
    g_two_pulse_2, phi_A0=0., phi_B0=0.05)
plot_two_pulse_coupled_osc(t_spikes_A, t_spikes_B, num_spikes_A, num_spikes_B)

In [ ]:
interact(lambda phi_B0=0.05: plot_two_pulse_coupled_osc(
    *simulate_two_pulse_coupled_osc(g_two_pulse_2, phi_A0=0., phi_B0=phi_B0)),
    phi_B0=(0.0, 1.0, 0.05));

## RTM Phase-Resetting Curve and Two-Event Map

`rtm_init` finds the single-cell RTM limit cycle (plain Heun integration
until the 5th spike) and interpolates $(v,h,n)$ at each requested phase --
the same splay-initialization recipe as Chapter 25's PRC examples.

In [ ]:
@njit
def _rtm_alpha_h(v):
    return 0.128 * np.exp(-(v + 50.0) / 18.0)


@njit
def _rtm_alpha_m(v):
    return 0.32 * (v + 54.0) / (1.0 - np.exp(-(v + 54.0) / 4.0))


@njit
def _rtm_alpha_n(v):
    return 0.032 * (v + 52.0) / (1.0 - np.exp(-(v + 52.0) / 5.0))


@njit
def _rtm_beta_h(v):
    return 4.0 / (1.0 + np.exp(-(v + 27.0) / 5.0))


@njit
def _rtm_beta_m(v):
    return 0.28 * (v + 27.0) / (np.exp((v + 27.0) / 5.0) - 1.0)


@njit
def _rtm_beta_n(v):
    return 0.5 * np.exp(-(v + 57.0) / 40.0)


@njit
def _rtm_m_inf(v):
    am, bm = _rtm_alpha_m(v), _rtm_beta_m(v)
    return am / (am + bm)


@njit
def _rtm_h_inf(v):
    ah, bh = _rtm_alpha_h(v), _rtm_beta_h(v)
    return ah / (ah + bh)


@njit
def _rtm_n_inf(v):
    an, bn = _rtm_alpha_n(v), _rtm_beta_n(v)
    return an / (an + bn)


@njit
def tau_peak_function(tau_d, tau_r, tau_d_q):
    '''Time (from a delta-function pulse of transmitter release) at which
    the synaptic gate s peaks.'''
    dt = 0.01
    dt05 = dt / 2
    s, t = 0.0, 0.0
    s_inc = np.exp(-t / tau_d_q) * (1 - s) / tau_r - s * tau_d
    while s_inc > 0:
        t_old, s_inc_old = t, s_inc
        s_tmp = s + dt05 * s_inc
        s_inc_tmp = np.exp(-(t + dt05) / tau_d_q) * (1 - s_tmp) / tau_r - s_tmp / tau_d
        s = s + dt * s_inc_tmp
        t = t + dt
        s_inc = np.exp(-t / tau_d_q) * (1 - s) / tau_r - s / tau_d
    return (t_old * (-s_inc) + t * s_inc_old) / (s_inc_old - s_inc)


@njit
def tau_d_q_function(tau_d, tau_r, tau_hat):
    '''Release time constant tau_d_q so that tau_peak_function reproduces
    the prescribed tau_hat (bisection, since there's no closed form).'''
    tau_d_q_left = 1.0
    while tau_peak_function(tau_d, tau_r, tau_d_q_left) > tau_hat:
        tau_d_q_left /= 2
    tau_d_q_right = tau_r
    while tau_peak_function(tau_d, tau_r, tau_d_q_right) < tau_hat:
        tau_d_q_right *= 2
    while tau_d_q_right - tau_d_q_left > 1e-12:
        tau_d_q_mid = (tau_d_q_left + tau_d_q_right) / 2
        if tau_peak_function(tau_d, tau_r, tau_d_q_mid) <= tau_hat:
            tau_d_q_left = tau_d_q_mid
        else:
            tau_d_q_right = tau_d_q_mid
    return (tau_d_q_left + tau_d_q_right) / 2


@njit
def rtm_init(i_ext, phi_vec, c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
             v_k=-100.0, v_na=50.0, v_l=-67.0, t_final=5000.0, dt=0.001, v0=-70.0):
    '''Find the RTM limit cycle at i_ext (plain-float Heun integration
    until the 5th spike), then interpolate (v, h, n) at each phase in
    phi_vec (fraction of the last full period, measured from the 4th
    spike). Returns (len(phi_vec), 3) array of initial conditions and
    the period T (np.inf if i_ext is subthreshold).'''
    dt05 = dt / 2

    v = [v0]
    m = [_rtm_m_inf(v0)]
    h = [_rtm_h_inf(v0)]
    n = [_rtm_n_inf(v0)]
    t_spikes = []

    k = 0
    t = 0.0
    while len(t_spikes) < 5 and t < t_final:
        vk, mk, hk, nk = v[k], m[k], h[k], n[k]
        v_inc = (g_k * nk ** 4 * (v_k - vk) + g_na * mk ** 3 * hk * (v_na - vk)
                 + g_l * (v_l - vk) + i_ext) / c
        h_inc = _rtm_alpha_h(vk) * (1 - hk) - _rtm_beta_h(vk) * hk
        n_inc = _rtm_alpha_n(vk) * (1 - nk) - _rtm_beta_n(vk) * nk

        v_tmp = vk + dt05 * v_inc
        m_tmp = _rtm_m_inf(v_tmp)
        h_tmp = hk + dt05 * h_inc
        n_tmp = nk + dt05 * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = _rtm_alpha_h(v_tmp) * (1 - h_tmp) - _rtm_beta_h(v_tmp) * h_tmp
        n_inc = _rtm_alpha_n(v_tmp) * (1 - n_tmp) - _rtm_beta_n(v_tmp) * n_tmp

        v.append(vk + dt * v_inc)
        h.append(hk + dt * h_inc)
        n.append(nk + dt * n_inc)
        m.append(_rtm_m_inf(v[-1]))

        if vk >= -20 and v[-1] < -20:
            t_spike = ((k) * dt * (-20 - v[-1]) + (k + 1) * dt * (20 + vk)) / (vk - v[-1])
            t_spikes.append(t_spike)
        t = (k + 1) * dt
        k += 1

    num = len(phi_vec)
    if len(t_spikes) < 5:
        v_last, h_last, n_last = v[k], h[k], n[k]
        out = np.zeros((num, 3))
        for i in range(num):
            out[i, 0], out[i, 1], out[i, 2] = v_last, h_last, n_last
        return out, np.inf

    T = t_spikes[4] - t_spikes[3]
    out = np.zeros((num, 3))
    for i, phi0 in enumerate(phi_vec):
        t0 = phi0 * T + t_spikes[3]
        kk = int(t0 / dt)
        frac_hi = (t0 - kk * dt) / dt
        frac_lo = ((kk + 1) * dt - t0) / dt
        out[i, 0] = v[kk + 1] * frac_hi + v[kk] * frac_lo
        out[i, 1] = h[kk + 1] * frac_hi + h[kk] * frac_lo
        out[i, 2] = n[kk + 1] * frac_hi + n[kk] * frac_lo
    return out, T

`simulate_rtm_g` splay-initializes RTM neurons across every phase and
delivers a single synaptic pulse to all of them at time 0 -- the same
recipe as Chapter 25's `simulate_synaptic_pulse_prc`, here measuring the
PRC $g(\varphi)$ used for comparison with the abstract maps above.
`compute_bigG_from_g` then folds that numerically measured PRC into the
two-event map $G$ by composing $F=f(1-\cdot)$ with itself via linear
interpolation, matching `pulse_map_bigG` above but working from a discrete
grid instead of a closed-form $g$. `simulate_rtm_plot_g` chains the two.

In [ ]:
def simulate_rtm_g(N=200, g_syn=0.1, i_ext=0.30, dt=0.01, tau_r=0.5, tau_peak=0.5, tau_d=2.0,
                    c=1.0, g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0):
    tau_dq = tau_d_q_function(tau_d, tau_r, tau_peak)
    dt05 = dt / 2

    phi_vec = np.arange(1, N) / N
    initial_vector, T = rtm_init(i_ext, phi_vec, c=c, g_k=g_k, g_na=g_na, g_l=g_l,
                                  v_k=v_k, v_na=v_na, v_l=v_l)

    v = initial_vector[:, 0].copy()
    m = _rtm_m_inf(v)
    h = initial_vector[:, 1].copy()
    n = initial_vector[:, 2].copy()
    q = np.ones(N - 1)
    s = np.zeros(N - 1)

    t_star = np.full(N - 1, np.nan)
    num_spikes = np.zeros(N - 1, dtype=int)

    k = 0
    while np.min(num_spikes) < 1:
        k += 1

        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v)
                 - g_syn * s * v + i_ext) / c
        h_inc = _rtm_alpha_h(v) * (1 - h) - _rtm_beta_h(v) * h
        n_inc = _rtm_alpha_n(v) * (1 - n) - _rtm_beta_n(v) * n
        q_inc = -q / tau_dq
        s_inc = q * (1 - s) / tau_r - s / tau_d

        v_tmp = v + dt05 * v_inc
        m_tmp = _rtm_m_inf(v_tmp)
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc
        q_tmp = q + dt05 * q_inc
        s_tmp = s + dt05 * s_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) - g_syn * s_tmp * v_tmp + i_ext) / c
        h_inc = _rtm_alpha_h(v_tmp) * (1 - h_tmp) - _rtm_beta_h(v_tmp) * h_tmp
        n_inc = _rtm_alpha_n(v_tmp) * (1 - n_tmp) - _rtm_beta_n(v_tmp) * n_tmp
        q_inc = -q_tmp / tau_dq
        s_inc = q_tmp * (1 - s_tmp) / tau_r - s_tmp / tau_d

        v_old = v.copy()
        v = v + dt * v_inc
        m = _rtm_m_inf(v)
        h = h + dt * h_inc
        n = n + dt * n_inc
        q = q + dt * q_inc
        s = s + dt * s_inc

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for i in ind:
            if num_spikes[i] == 0:
                t_star[i] = ((k - 1) * dt * (-20 - v[i]) + k * dt * (v_old[i] + 20)) / (v_old[i] - v[i])
            num_spikes[i] += 1

    g_vec = -t_star / T + 1 - phi_vec
    return phi_vec, g_vec, T


def compute_bigG_from_g(phi_vec, g_vec, N):
    '''Fold a numerically measured PRC g(phi) into the two-event map G,
    via F=f(1-.) (f=phi+g) composed with itself, by linear interpolation
    on the phi-grid.'''
    dphi = 1 / N
    phi_vec = np.concatenate(([0.], phi_vec, [1.]))
    g_vec = np.concatenate(([0.], g_vec, [0.]))
    f_vec = phi_vec + g_vec
    bigF_vec = f_vec[::-1]
    bigG_vec = np.zeros(N + 1)
    bigG_vec[N] = 1.

    for k in range(1, N):
        F0 = bigF_vec[k]
        i = int(np.floor(F0 / dphi))
        bigG_vec[k] = (bigF_vec[i] * ((i + 1) * dphi - F0) + bigF_vec[i + 1] * (F0 - i * dphi)) / dphi

    return phi_vec, bigG_vec


def simulate_rtm_plot_g(N=200, g_syn=0.1, i_ext=0.30, dt=0.01, tau_r=0.5, tau_peak=0.5, tau_d=2.0,
                         c=1.0, g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0):
    phi_vec, g_vec, T = simulate_rtm_g(N=N, g_syn=g_syn, i_ext=i_ext, dt=dt, tau_r=tau_r, tau_peak=tau_peak,
                                        tau_d=tau_d, c=c, g_k=g_k, g_na=g_na, g_l=g_l, v_k=v_k, v_na=v_na, v_l=v_l)
    phi_vec, bigG_vec = compute_bigG_from_g(phi_vec, g_vec, N)
    return phi_vec, bigG_vec, T


def plot_rtm_plot_g(phi_vec, bigG_vec):
    plt.figure(figsize=(6, 6))
    plt.plot(phi_vec, bigG_vec, '-k', linewidth=6)
    plt.axis([0, 1, 0, 1])
    plt.gca().set_box_aspect(1)
    plt.xlabel(r'$\varphi$')
    plt.ylabel('$G$')
    plt.plot([0, 1], [0, 1], '--k', linewidth=2)
    plt.tight_layout()
    plt.show()

In [ ]:
phi_vec, bigG_vec, T = simulate_rtm_plot_g()
plot_rtm_plot_g(phi_vec, bigG_vec)

In [ ]:
interact(lambda i_ext=0.30: plot_rtm_plot_g(*simulate_rtm_plot_g(i_ext=i_ext)[:2]),
         i_ext=(0.20, 0.50, 0.01));